### Create the LunarLander environment

In [1]:
import gym
import numpy as np
import pandas as pd


env = gym.make("LunarLander-v2")

### Setup R packages

In [2]:
from cxrl.lib.r_to_py import setup_R
setup_R()

[]


### A sample policy for LunarLander

In [3]:
def policy(state):
    """
    Simple heuristic policy for Gym/Gymnasium LunarLander with discrete actions.

    state:
        x, y, vx, vy, angle, angular_velocity, left_contact, right_contact

    returns:
        action in {0, 1, 2, 3}
    """
    x, y, vx, vy, angle, angular_velocity, left_contact, right_contact = state

    # 1. Point the lander toward the center.
    # If x > 0, lander is right of pad, so target a positive angle
    # to move back left. vx helps damp horizontal motion.
    target_angle = 0.5 * x + 1.0 * vx
    target_angle = np.clip(target_angle, -0.4, 0.4)

    # 2. When far from center, stay higher; when near center, descend.
    target_y = 0.55 * abs(x)

    # 3. PD-style control errors.
    angle_control = 0.5 * (target_angle - angle) - 1.0 * angular_velocity
    hover_control = 0.5 * (target_y - y) - 0.5 * vy

    # 4. Once a leg touches, stop trying to rotate; just reduce falling speed.
    if left_contact or right_contact:
        angle_control = 0.0
        hover_control = -0.5 * vy

    # 5. Convert desired controls to discrete actions.
    if hover_control > abs(angle_control) and hover_control > 0.05:
        return 2      # main engine
    elif angle_control < -0.05:
        return 3      # right orientation engine
    elif angle_control > 0.05:
        return 1      # left orientation engine
    else:
        return 0      # do nothing

### Infer the global causal graph from episodes

Proposed method elapsed time

In [9]:
from cxrl.lib.xrl import XRL

# Names for interpretable representations: 8 state variables + 4 one-hot actions.
state_names = [
    'x', 'y', 'vx', 'vy', 'angle', 'angular_velocity',
    'left_contact', 'right_contact'
]
action_names = [
    'do_nothing', 'fire_left_engine', 'fire_main_engine', 'fire_right_engine'
]
feature_names = state_names + action_names

# LunarLander timing is heavier than CartPole because it has more variables and
# longer episodes. Increase NEPISODES for a larger benchmark run.
NEPISODES = 20
xrl = XRL(
    env,
    pi=policy,
    repr_names=feature_names,
    verbose=True,
    one_hot_actions=True,
    repr_integration=False,
    nepisodes=NEPISODES,
    method='expectation',
)

experience size= 6176
 causes: ['x', 'angle'] -> x
 causes: ['y', 'left_contact'] -> y
 causes: ['vx', 'fire_right_engine'] -> vx
 causes: ['vy'] -> vy
 causes: ['angle'] -> angle
 causes: ['angular_velocity', 'left_contact'] -> angular_velocity
 causes: [] -> left_contact
 causes: [] -> right_contact
 causes: [] -> do_nothing
 causes: [] -> fire_left_engine
 causes: [] -> fire_main_engine
 causes: [] -> fire_right_engine
BART SLA elapsed time: 0.320850133895874 seconds


In [15]:
#graph include action
INCLUDE_ACTION = False
states, actions, rewards, states_new = xrl.replay_buffer

from cxrl.lib.utils import convert_and_expand
states = convert_and_expand(states).astype(float)

if INCLUDE_ACTION:
    actions = convert_and_expand(actions).astype(int)

    # Build fixed-width one-hot actions so the baseline timing data has the same
    # representation columns used by XRL.
    action_one_hot = np.zeros((actions.shape[0], len(action_names)), dtype=float)
    action_one_hot[np.arange(actions.shape[0]), actions.squeeze()] = 1.0

    sla_data = np.concatenate((states, action_one_hot), axis=1)
else:
    sla_data = states
print('SLA data shape:', sla_data.shape)

SLA data shape: (6176, 8)


### Customized CI test embedded simulations of BART CI test

In [16]:
from causallearn.utils.cit import CIT, CIT_Base, register_ci_test

from sklearn.model_selection import train_test_split
import time
import warnings
warnings.filterwarnings('ignore')


class CustomCIT(CIT_Base):
    total_time = 0.0

    @classmethod
    def reset_timer(cls):
        cls.total_time = 0.0

    @classmethod
    def get_total_time(cls):
        return cls.total_time

    def __init__(self, data, **kwargs):
        super().__init__(data, **kwargs)
        self.method = "custom_test"

        # Important: do not recreate Fisher-Z inside every CI call
        self.fisherz_obj = CIT(self.data, "fisherz")

    def __call__(self, X, Y, S=None):
        if S is None:
            S = []

        X_train, X_test, y_train, y_test = train_test_split(
            self.data,
            self.data[:, Y],
            test_size=0.5
        )

        y_hat_cf = (np.random.rand(500, y_test.shape[0]) - 0.5) * 10

        start_time = time.time()

        # simulation block
        sse_cf = (y_test - y_hat_cf) ** 2
        mse_cf = sse_cf.mean(axis=1)
        _, _ = np.quantile(mse_cf, [0.05, 0.95])
        _ = np.quantile(sse_cf, 0.95)

        elapsed_time = time.time() - start_time
        CustomCIT.total_time += elapsed_time

        # actual CI test
        p_value = self.fisherz_obj(X, Y, S)
        return p_value


# Register the CI test
register_ci_test("custom_test", CustomCIT)

if INCLUDE_ACTION:
    sla_data = np.concatenate((states, action_one_hot), axis=1)
else:
    sla_data = states

### Customized score function

In [17]:
from causallearn.search.ScoreBased import GES as ges_module
from causallearn.score.LocalScoreFunction import local_score_BIC as original_local_score_BIC

class CustomGESScore:
    total_time = 0.0

    @classmethod
    def reset_timer(cls):
        cls.total_time = 0.0

    @classmethod
    def get_total_time(cls):
        return cls.total_time


    @staticmethod
    def local_score(Data, i, PAi, parameters=None):
        """
        Customized local score function for GES.

        Data: numpy array, shape = (n_samples, n_features)
        i: target variable index
        PAi: list of parent indices for variable i
        parameters: optional score parameters

        Must return a scalar score. Higher score is better in causal-learn GES.
        """

        PAi = list(PAi)

        # simulation block
        _, X_test, _, y_test = train_test_split(
            Data,
            Data[:, i],
            test_size=0.5
        )

        y_hat_cf = (np.random.rand(500, y_test.shape[0]) - 0.5) * 10

        start_time = time.time()

        sse_cf = (y_test - y_hat_cf) ** 2
        mse_cf = sse_cf.mean(axis=1)
        _, _ = np.quantile(mse_cf, [0.05, 0.95])
        _ = np.quantile(sse_cf, 0.95)

        elapsed_time = time.time() - start_time

        CustomGESScore.total_time += elapsed_time

        # actual bic score
        score = original_local_score_BIC(Data, i, PAi, parameters)

        return score


def ges_with_custom_bic_score(X, maxP=None, parameters=None, node_names=None):
    """
    Temporarily replace causal-learn's local_score_BIC with the custom score,
    run GES, then restore the original functions.
    """

    # Save originals
    original_ges_local_score_BIC = ges_module.local_score_BIC

    # Some causal-learn versions internally route BIC through
    # local_score_BIC_from_cov, so save and patch it if it exists.
    has_bic_from_cov = hasattr(ges_module, "local_score_BIC_from_cov")
    if has_bic_from_cov:
        original_ges_local_score_BIC_from_cov = ges_module.local_score_BIC_from_cov

    try:
        # Patch BIC score used inside GES
        ges_module.local_score_BIC = CustomGESScore.local_score

        if has_bic_from_cov:
            ges_module.local_score_BIC_from_cov = CustomGESScore.local_score

        # Run GES. Keep score_func as "local_score_BIC".
        Record = ges_module.ges(
            X,
            score_func="local_score_BIC",
            maxP=maxP,
            parameters=parameters,
            node_names=node_names
        )

    finally:
        # Always restore original functions
        ges_module.local_score_BIC = original_ges_local_score_BIC

        if has_bic_from_cov:
            ges_module.local_score_BIC_from_cov = original_ges_local_score_BIC_from_cov

    return Record

### Baseline elapsed times

In [18]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.search.ScoreBased.GES import ges

# ---------- PC algorithm ----------
print("Learn causal graph using PC algorithm...")
CustomCIT.reset_timer()

cg = pc(
    sla_data,
    alpha=0.05,
    indep_test="custom_test",
    stable=False
)

pc_ci_time = CustomCIT.get_total_time()
print(f"* PC algorithm SLA elapsed CI-test time: {pc_ci_time:.4f} seconds\n")


# ---------- FCI algorithm ----------
print("Learn causal graph using FCI algorithm...")
CustomCIT.reset_timer()

g, edges = fci(
    sla_data,
    independence_test_method="custom_test",
    alpha=0.05,
    verbose=False
)

fci_ci_time = CustomCIT.get_total_time()
print(f"* FCI algorithm SLA elapsed CI-test time: {fci_ci_time:.4f} seconds\n")



# ---------- GES algorithm ----------
print("Learn causal graph using GES algorithm...")
CustomGESScore.reset_timer()

Record = ges_with_custom_bic_score(
    sla_data,
    maxP=None
)

print(f"* GES algorithm SLA elapsed score-computation time: "
      f"{CustomGESScore.get_total_time():.4f} seconds")

Learn causal graph using PC algorithm...


Depth=4, working on node 7: 100%|██████████| 8/8 [00:00<00:00, 11.84it/s]  


* PC algorithm SLA elapsed CI-test time: 12.5739 seconds

Learn causal graph using FCI algorithm...


Depth=0, working on node 7: 100%|██████████| 8/8 [00:01<00:00,  5.91it/s]


X1 --> X5
X6 --> X5
* FCI algorithm SLA elapsed CI-test time: 25.5393 seconds

Learn causal graph using GES algorithm...
* GES algorithm SLA elapsed score-computation time: 3.9616 seconds
